# MAP estimation exploration

Visual walk-through of `bayes_cep.optimization`: builds the `LogPosterior` for the single retained patient (`example_data/raw/`, `example_data/preprocessing/`), then finds its MAP estimate via `CustomLBFGSStrategy` -- L-BFGS in the prior's Cameron-Martin space, metric-consistent with the prior (see `bayes_cep.optimization.model`'s docstring). `ScipyLBFGSBStrategy`, plain Euclidean L-BFGS-B via `scipy.optimize.minimize`, is the non-metric-consistent alternative.

This notebook is for interactive/visual inspection only. `pixi run python scripts/run_map.py --data-dir example_data` runs the same pipeline non-interactively and caches its outputs to `example_data/map/` (see that script's docstring, or `--help`, for the full set of tunable parameters, including how to switch to `ScipyLBFGSBStrategy`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
from ls_bayesian.common.logging import BaseLogger, LoggerSettings
from ls_bayesian.optimization.algorithms.custom_lbfgs import CustomLBFGSSettings

from bayes_cep.mesh.interpolation import NearestNeighborInterpolationStrategy
from bayes_cep.mesh.io import load_pyvista_mesh
from bayes_cep.optimization.strategies import CustomLBFGSStrategy
from bayes_cep.posterior.builder import PosteriorBuilder, PosteriorSettings
from bayes_cep.posterior.eikonal_map import EikonalSolverSettings
from bayes_cep.posterior.likelihood import LikelihoodSettings
from bayes_cep.posterior.prior import PriorSettings
from bayes_cep.preprocessing.axial_statistics import compute_axial_data_diff

pv.set_jupyter_backend("trame")

DATA_DIR = Path.cwd().parent / "example_data"
RAW_DIR = DATA_DIR / "raw"
PREPROCESSING_DIR = DATA_DIR / "preprocessing"

## Load preprocessed data

In [ ]:
mesh = load_pyvista_mesh(RAW_DIR / "mesh.vtu")
basis_vectors = np.load(RAW_DIR / "basis_vecs.npy")
ground_truth_angle_field = np.load(PREPROCESSING_DIR / "ground_truth_angle_field.npy")
prior_mean_angle_field = np.load(PREPROCESSING_DIR / "prior_mean_angle_field.npy")
observed_vertex_indices = np.load(PREPROCESSING_DIR / "observed_vertex_indices.npy")
observed_activation_times = np.load(PREPROCESSING_DIR / "observed_activation_times.npy")

print(f"{mesh.n_points} vertices, {mesh.n_cells} triangles")
print(f"{observed_vertex_indices.shape[0]} observed vertices")

## Ground truth

The parameter is a fiber-angle value per mesh vertex, but the forward model (and, correspondingly,
the comparisons below) only ever sees it interpolated onto simplices, via the same
`InterpolationStrategy` (`NearestNeighborInterpolationStrategy` here, matching
`EikonalParameterToSolutionMap`'s default) used to build `vertex_to_simplex_matrix` below. All
angle fields are therefore visualized and compared on simplices (`cell_data`), not vertices
(`point_data`), from here on.

In [ ]:
interpolation_strategy = NearestNeighborInterpolationStrategy()
connectivity = mesh.cells.reshape(-1, 4)[:, 1:]
vertex_to_simplex_matrix = interpolation_strategy.assemble_matrix(mesh.points, connectivity)

ground_truth_angle_field_simplex = vertex_to_simplex_matrix @ ground_truth_angle_field

plot_mesh = mesh.copy()
plot_mesh.cell_data["angle"] = ground_truth_angle_field_simplex
plotter = pv.Plotter()
plotter.add_mesh(plot_mesh, scalars="angle", cmap="hsv")
plotter.add_text("ground truth")
plotter.show()

## Build the posterior

Reference settings, matching `scripts/run_map.py`'s and `scripts/preprocess_data.py`'s defaults.
`interpolation_strategy` is the same object used above for `vertex_to_simplex_matrix`, so the
posterior's forward map and the ground-truth/MAP comparisons agree on how the parameter maps onto
simplices.

In [ ]:
posterior_settings = PosteriorSettings(
    mesh=mesh,
    basis_vectors=basis_vectors,
    prior_settings=PriorSettings(
        mean_vector=prior_mean_angle_field,
        kappa=0.05,
        tau=10.0,
        seed=0,
    ),
    eikonal_settings=EikonalSolverSettings(
        initial_site_ind=12650,
        longitudinal_velocity=1.5,
        transversal_velocity=1.0,
    ),
    likelihood_settings=LikelihoodSettings(
        num_vertices=mesh.points.shape[0],
        observed_vertex_indices=observed_vertex_indices,
        data_vector=observed_activation_times,
        noise_variance=1e-3,
    ),
)
posterior_builder = PosteriorBuilder(
    posterior_settings, interpolation_strategy=interpolation_strategy
)
log_posterior = posterior_builder.build()

## Run MAP estimation

`CustomLBFGSStrategy` runs L-BFGS in the prior's Cameron-Martin space (metric-consistent, with Armijo backtracking by default and cautious updating) and is paired with `CameronMartinPosteriorModel`. `ScipyLBFGSBStrategy`, which wraps `scipy.optimize.minimize(method="L-BFGS-B")` under the standard Euclidean inner product, is available as a drop-in alternative by building `ScipyLBFGSBStrategy(...)` instead.

Besides the L-BFGS memory size, the convergence tolerance on the gradient norm (`gradient_norm_tolerance=100.0`, much looser than the library default of `1e-6`) and the iteration cap (`maximum_num_iterations=1000`, the library default) are set here, matching `scripts/run_map.py`. The strategy's default seed scaling (`gamma_min=1e-10`, first-iteration `fallback_value=1e-6`, see `bayes_cep.optimization.strategies`) is tuned to this problem: it avoids about a dozen backtracking forward solves per iteration, and about 20 more in iteration 1. Each iteration solves the eikonal forward problem and its adjoint once, so lower it for a quick run, and check `success` and `status_message` below to see whether the optimizer converged or stopped at the cap.

> **Note:** the optimization below can take a long time (each iteration needs a forward and an adjoint eikonal solve on the full mesh, and up to 1000 iterations are allowed by default). To only inspect the results, set `LOAD_SAVED_MAP = True` in the next cell: it skips the optimization and loads the MAP estimate and convergence histories cached in `example_data/map/` instead (produced by `scripts/run_map.py --data-dir example_data`; they reflect that script's settings, which may differ from the ones in this notebook).

In [ ]:
# True: skip the (slow) optimization and load the cached example results from `example_data/map/`.
LOAD_SAVED_MAP = True

if LOAD_SAVED_MAP:
    MAP_DIR = DATA_DIR / "map"
    map_estimate = np.load(MAP_DIR / "map_estimate.npy")
    loss_history = np.load(MAP_DIR / "map_loss_history.npy")
    gradient_norm_history = np.load(MAP_DIR / "map_gradient_norm_history.npy")
    print(f"loaded cached MAP estimate from {MAP_DIR}")
else:
    strategy = CustomLBFGSStrategy(
        lbfgs_settings=CustomLBFGSSettings(
            memory_size=10, maximum_num_iterations=1000, gradient_norm_tolerance=100.0
        )
    )
    with BaseLogger(LoggerSettings(), prefix="map") as logger:
        model, optimizer = strategy.build(log_posterior, posterior_builder.prior, logger)
        result = optimizer.run(initial_guess=prior_mean_angle_field, model=model)

    map_estimate = result.result
    loss_history = result.loss_history
    gradient_norm_history = result.gradient_norm_history
    print(f"success: {result.success}")
    print(f"status: {result.status_message}")

## Convergence history

In [ ]:
fig, (loss_axis, gradient_axis) = plt.subplots(1, 2, figsize=(10, 4))
loss_axis.semilogy(loss_history, marker="o")
loss_axis.set_xlabel("iteration")
loss_axis.set_ylabel("loss")
gradient_axis.semilogy(gradient_norm_history, marker="o")
gradient_axis.set_xlabel("iteration")
gradient_axis.set_ylabel("Cameron-Martin gradient norm")
fig.tight_layout()

## MAP estimate vs. ground truth and prior mean

Compared on simplices (via `vertex_to_simplex_matrix`, see "Ground truth" above), not vertices:
this is the representation the forward model actually sees, and what `ground_truth_angle_field`
looks like averaged onto simplices is already shown above.

In [ ]:
prior_mean_angle_field_simplex = vertex_to_simplex_matrix @ prior_mean_angle_field
map_estimate_simplex = vertex_to_simplex_matrix @ map_estimate

# Ground truth and MAP estimate share one color range, spanning both fields.
shared_clim = (
    min(ground_truth_angle_field_simplex.min(), map_estimate_simplex.min()),
    max(ground_truth_angle_field_simplex.max(), map_estimate_simplex.max()),
)

for name, field_values, clim in (
    ("prior mean", prior_mean_angle_field_simplex, None),
    ("ground truth", ground_truth_angle_field_simplex, shared_clim),
    ("MAP estimate", map_estimate_simplex, shared_clim),
):
    plot_mesh = mesh.copy()
    plot_mesh.cell_data["angle"] = field_values
    plotter = pv.Plotter()
    plotter.add_mesh(plot_mesh, scalars="angle", cmap="hsv", clim=clim)
    plotter.add_text(name)
    plotter.show()

## Axial error vs. ground truth

Since the fiber angle is $\pi$-periodic, the pointwise error against the ground truth is the signed axial difference (`compute_axial_data_diff`), not a plain subtraction: comparing the MAP estimate's mean absolute error to the prior mean's is a sanity check that optimization actually moved towards the data. Computed on the simplex-interpolated fields, matching the comparison above.

In [ ]:
prior_mean_error = compute_axial_data_diff(
    prior_mean_angle_field_simplex, ground_truth_angle_field_simplex
)
map_estimate_error = compute_axial_data_diff(map_estimate_simplex, ground_truth_angle_field_simplex)
print(f"prior mean mean absolute error:  {np.mean(np.abs(prior_mean_error)):.4f} rad")
print(f"MAP estimate mean absolute error: {np.mean(np.abs(map_estimate_error)):.4f} rad")

for name, error_field in (
    ("prior mean error", prior_mean_error),
    ("MAP estimate error", map_estimate_error),
):
    plot_mesh = mesh.copy()
    plot_mesh.cell_data["error"] = error_field
    plotter = pv.Plotter()
    plotter.add_mesh(plot_mesh, scalars="error", cmap="coolwarm", clim=(-np.pi / 2, np.pi / 2))
    plotter.add_text(name)
    plotter.show()

## Arrival-time differences: true vs. prior, MAP vs. prior

The comparisons above are all in parameter space (the fiber angle). This one compares in the space
the likelihood actually sees: the eikonal forward solve's arrival-time (activation-time) field, via
`posterior_builder.forward_map` -- the same forward map the posterior was built with.
`true - prior` is the baseline arrival-time mismatch the observations have to correct for;
`map - prior` is how far optimization actually moved the arrival times away from that baseline --
ideally in a similar direction and of comparable magnitude to `true - prior`.

In [ ]:
true_arrival_times = posterior_builder.forward_map.evaluate_forward(ground_truth_angle_field)
prior_arrival_times = posterior_builder.forward_map.evaluate_forward(prior_mean_angle_field)
map_arrival_times = posterior_builder.forward_map.evaluate_forward(map_estimate)

true_minus_prior = true_arrival_times - prior_arrival_times
true_minus_map = true_arrival_times - map_arrival_times
max_abs_diff = max(np.max(np.abs(true_minus_prior)), np.max(np.abs(true_minus_map)))

print(f"mean |true-prior| arrival-time diff: {np.mean(np.abs(true_minus_prior)):.4f}")
print(f"mean |true-map|  arrival-time diff: {np.mean(np.abs(true_minus_map)):.4f}")

for name, diff_field in (
    ("true - prior", true_minus_prior),
    ("true - map", true_minus_map),
):
    plot_mesh = mesh.copy()
    plot_mesh.point_data["arrival_time_diff"] = diff_field
    plotter = pv.Plotter()
    plotter.add_mesh(
        plot_mesh, scalars="arrival_time_diff", cmap="coolwarm", clim=(-max_abs_diff, max_abs_diff)
    )
    plotter.add_text(name)
    plotter.show()